In [22]:
import pandas as pd
from pathlib import Path

In [23]:
DATA_PATH  = Path("../data/processed/primero_bank_clients_nettoyes.csv")

df = pd.read_csv(DATA_PATH) 

df.head()

,N° du client,Statut du client,Âge du client,Genre du client,Nb de personnes à charge,Niveau de diplôme,Statut marital,Catégorie du revenu annuel,Type de carte,Durée d'engagement en mois,Nb de mois inactif,Nb d'interactions,Montant crédit renouvellé,Nb de transactions,Utilisation moyenne de la carte
0,768805383,Client actuel,45,M,3,Lycée (équivalent baccalauréat),Marié(e),$60K - $80K,Blue,39,1,3,777,42,0.061
1,818770008,Client actuel,49,F,5,Licence,Célibataire,Moins de $40K,Blue,44,1,2,864,33,0.105
2,713982108,Client actuel,51,M,3,Licence,Marié(e),$80K - $120K,Blue,36,1,0,0,20,0.000
3,769911858,Client actuel,40,F,4,Lycée (équivalent baccalauréat),Non connu,Moins de $40K,Blue,34,4,1,2517,20,0.760
4,709106358,Client actuel,40,M,3,Sans diplôme,Marié(e),$60K - $80K,Blue,21,1,0,0,28,0.000


## On garde seulement les clients actuels 

In [24]:
clients_actuels = df[df["Statut du client"] == "Client actuel"].copy()

print("nombre de clients actuels : ", len (clients_actuels) ) 

nombre de clients actuels :  8491


In [34]:
clients_perdus = df[df["Statut du client"] == "Client perdu" ].copy() 

seuils = {"mois_inactifs" : clients_perdus["Nb de mois inactif"].mean() ,
         "interactions" : clients_perdus["Nb d'interactions"].mean() , 
         "credit": clients_perdus["Montant crédit renouvellé"].mean(),
          "transactions" : clients_perdus["Nb de transactions"].mean() , 
          "utilisation" : clients_perdus["Utilisation moyenne de la carte"].mean() 
            }

seuils 

{'mois_inactifs': np.float64(3.2334963325183375),
 'interactions': np.float64(3.480440097799511),
 'credit': np.float64(678.6466992665037),
 'transactions': np.float64(45.18154034229829),
 'utilisation': np.float64(0.16185696821515894)}

## Creons les indicateurs de risque 

In [35]:
clients_actuels["risque_inactivite"] = (
    clients_actuels["Nb de mois inactif"]
    >= seuils["mois_inactifs"]
).astype(int)

clients_actuels["risque_interactions"] = (
    clients_actuels["Nb d'interactions"]
    >= seuils["interactions"]
).astype(int)

clients_actuels["risque_credit"] = (
    clients_actuels["Montant crédit renouvellé"]
    <= seuils["credit"]
).astype(int)

clients_actuels["risque_transactions"] = (
    clients_actuels["Nb de transactions"]
    <= seuils["transactions"]
).astype(int)

clients_actuels["risque_utilisation"] = (
    clients_actuels["Utilisation moyenne de la carte"]
    <= seuils["utilisation"]
).astype(int)

## Calculons le score total 

In [36]:
colonnes_risque = [
    "risque_inactivite",
    "risque_interactions",
    "risque_credit",
    "risque_transactions",
    "risque_utilisation"
]

clients_actuels["score_risque"] = (
    clients_actuels[colonnes_risque].sum(axis=1)
)

## Calculons le niveau de risque 

In [37]:
def definir_niveau_risque(score):
    if score <= 1:
        return "Faible"
    elif score <= 3:
        return "Modéré"
    else:
        return "Élevé"

In [38]:
clients_actuels["niveau_risque"] = (
    clients_actuels["score_risque"]
    .apply(definir_niveau_risque)
)

## Regardons la repartition 

In [41]:
clients_actuels["niveau_risque"].value_counts()

niveau_risque
Faible    5757
Modéré    2673
Élevé       61
Name: count, dtype: int64

In [ ]:
(
    clients_actuels["niveau_risque"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

niveau_risque
Faible    67.80
Modéré    31.48
Élevé      0.72
Name: proportion, dtype: float64

## Affichons les clients prioritaires à contacter en fonction de leur niveau de risque et de leur score de risque.

In [45]:
clients_prioritaires= ( clients_actuels[clients_actuels["niveau_risque"] == "Élevé"
                                      ].sort_values(by="score_risque", ascending=False) ) 

clients_prioritaires[
    [
        "N° du client",
        "Âge du client",
        "Nb de mois inactif",
        "Nb d'interactions",
        "Montant crédit renouvellé",
        "Nb de transactions",
        "Utilisation moyenne de la carte",
        "score_risque",
        "niveau_risque"
    ]
].head(20)

,N° du client,Âge du client,Nb de mois inactif,Nb d'interactions,Montant crédit renouvellé,Nb de transactions,Utilisation moyenne de la carte,score_risque,niveau_risque
1002,778706508,43,5,4,0,30,0.0,5,Élevé
3143,716211258,59,5,4,0,43,0.0,5,Élevé
92,714107958,45,4,3,0,34,0.0,4,Élevé
664,769170558,53,4,3,0,41,0.0,4,Élevé
360,711643458,50,3,4,0,41,0.0,4,Élevé
884,715848408,61,6,1,0,36,0.0,4,Élevé
1082,715511658,49,1,4,0,27,0.0,4,Élevé
1154,716383683,39,3,4,0,28,0.0,4,Élevé
1155,713693958,38,3,4,0,29,0.0,4,Élevé
1186,798110358,64,2,4,0,35,0.0,4,Élevé


## Exportons le resultat pour power bi 

In [46]:
OUTPUT_PATH = Path(
    "../data/processed/primero_bank_clients_avec_risque.csv"
)

clients_actuels.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Fichier exporté :", OUTPUT_PATH.resolve())

Fichier exporté : C:\Users\fayel\OneDrive\github\data\data-analyse-clients-banque\data\processed\primero_bank_clients_avec_risque.csv
